# Časť 2, Téma 1, Lab A: Rozdiely v spotrebe inštrukcií (MAIN)

---
POZNÁMKA: Toto cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke ChipWhisperer.io. Cvičenie môžete voľne spúšťať a používať v súlade s licenciou open source (vrátane použitia vo vašich vlastných kurzoch, ak ich distribuujete podobným spôsobom), musíte však zachovať odkaz na tento zdroj. Zvážte účasť na našom školiacom kurze, aby ste si mohli vychutnať plnohodnotný zážitok.

---

**Zhrnutie:** *Teraz, keď ste sa zoznámili s platformou ChipWhisperer, budeme ju využívať na to, aby sme sa dozvedeli viac o tom, ako sa spotreba energie mikrokontroléra mení v závislosti od toho, aké operácie vykonáva a aké inštrukcie spúšťa.*

**Čo sa naučíme:**

* Zaznamenávanie spotreby energie pomocou ChipWhisperer
* Vytváranie súvislostí medzi rôznymi jednoduchými inštrukciami a záznamami spotreby energie

## Predpoklady

Počkajte! Než budete pokračovať, skontrolujte, či ste absolvovali nasledujúce tutoriály:

* ☑ Úvod do Jupyter Notebooku (mali by ste vedieť kresliť grafy a spúšťať bloky).
* ☑ Úvod do SCA101 (mali by ste mať predstavu o tom, ako spustiť verzie špecifické pre konkrétny hardvér).

Najskor skompilujeme a nahráme firmvér na cieľ

In [11]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'

In [12]:
%%bash
cd ../../firmware/mcu
mkdir -p simpleserial-base-lab2 && cp -r simpleserial-base/* $_
cd simpleserial-base-lab2

In [13]:
%%bash -s "$PLATFORM"
cd ../../firmware/mcu/simpleserial-base-lab2
make PLATFORM=$1 CRYPTO_TARGET=NONE

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
avr-gcc (Homebrew AVR GCC 9.5.0) 9.5.0
Copyright (C) 2019 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEXMEGA 
.
Compiling:
-en     simpleserial-base.c ...
-e Done!
.
Compiling:
-en     .././simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     .././hal/hal.c ...
-e Done!
.
Compiling:
-en     .././hal//xmega/XMEGA_AES_driver.c ...
-e Done!
.
Compiling:
-en     .././hal//xmega/uart.c ...
-e Done!
.
Compiling:
-en     .././hal//xmega/usart_driver.c ...
-e Done!
.
Compiling:
-en     .././hal//xmega/xmega_hal.c ...
-e Done!
.
LINKING:
-en     simpleserial-base-CWLITEXMEGA.elf ...
Memory region         Used Size  Region Size  %age Used
            text:        2116 B         1 MB      0.20%
            data:         210

In [14]:
import chipwhisperer as cw
try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()
   
try:
    target = cw.target(scope)
except IOError:
    print("INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.")
    print("INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.")
    scope = cw.scope()
    target = cw.target(scope)

print("INFO: ChipWhisperer Nájdený😍")

if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
    prog = cw.programmers.XMEGAProgrammer
elif "neorv32" in PLATFORM.lower():
    prog = cw.programmers.NEORV32Programmer
elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
    prog = cw.programmers.SAM4SProgrammer
else:
    prog = None
    
import time
time.sleep(0.05)
scope.default_setup()

if PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
    scope.io.target_pwr = 0
    time.sleep(0.2)
    scope.io.target_pwr = 1
    time.sleep(0.2)
def reset_target(scope):
    if PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
        scope.io.pdic = 'low'
        time.sleep(0.1)
        scope.io.pdic = 'high_z' #XMEGA doesn't like pdic driven high
        time.sleep(0.1) #xmega needs more startup time
    elif "neorv32" in PLATFORM.lower():
        raise IOError("Default iCE40 neorv32 build does not have external reset - reprogram device to reset")
    elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
        scope.io.nrst = 'low'
        time.sleep(0.25)
        scope.io.nrst = 'high_z'
        time.sleep(0.25)
    else:  
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.io.nrst = 'high_z'
        time.sleep(0.05)

INFO: ChipWhisperer Nájdený😍


In [15]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-base-lab2/simpleserial-base-{}.hex".format(PLATFORM))

XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2115 bytes


In [16]:
def capture_trace(_ignored=None):
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    return cw.capture_trace(scope, target, text).wave

In [17]:
wave = capture_trace()
print("✔️ OK to continue!")

✔️ OK to continue!


## „Prázdny“ záznam

Ako ste videli v predchádzajúcich cvičeniach, firmvér `simpleserial-base` odzrkadľuje to, čo mu pošleme. V tomto laboratóriu sa chceme zamerať skôr na jednoduché inštrukcie ako na sériovú komunikáciu. Otvorte súbor `simpleserial-base.c` v priečinku `simpleserial-base-lab2`, prejdite na funkciu `get_pt()` a odstráňte volanie `simpleserial_put()`.

Znovu skompilujte firmvér a nahrajte ho do cieľového zariadenia. Zachyťte stopu a pomocou nasledujúceho kódu ju môžete znázorniť graficky:

In [ ]:
for _ in range(400000):
    wave_12 = capture_trace()
cw.plot(wave_12)

In [ ]:
cw.plot(wave)

## Jednoduché inštrukcie

Na úvod vložíme niekoľko jednoduchých inštrukcií, aby sme zistili, či sú viditeľné v grafickom zobrazení spotreby. Hľadáme hlavne inštrukciu, ktorá sa vykoná v jednom cykle, čo bude závisieť od platformy, ktorú používate pre toto cvičenie. Pre zariadenie ARM skúste vložiť nasledujúci kód, ktorý vykoná 20 násobení. Je dôležité označiť `A` ako `volatile`, aby kompilátor neoptimalizoval všetky tieto inštrukcie:

```C
	volatile long int A = 0x2BAA;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;

	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
	A *= 2;
```
Pri XMEGA/AVR násobenie je drahšia inštukcia preto vlož tento kód:

```C

	volatile long int A = 0x2BAA;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;

	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	
	A += 2;
	A += 2;
	A += 2;
	A += 2;
	A += 2;
```

Následne to znovu zkompiluj, zachyť, a daj do grafu

In [ ]:
wave2 = capture_trace()
cw.plot(wave2)

Závisí od cieľa, ale nemusí na prvý pohľad vidieť rozdiel. Skús hodiť oba záznamy na jeden graf.

In [ ]:
cw.plot(wave) * cw.plot(wave2)

## Jednoduchá inštrukčná slučka

Možno vám príde divné, že sme namiesto 20-násobného kopírovania tej istej veci nepoužili jednoducho slučku. Napravte to tak, že tieto inštrukcie nahradíte slučkou. **Nezabudnite nastaviť premennú slučky ako volatile.**

wave3 = capture_trace()
cw.plot(wave3)

Prekvapuje vás, ako veľmi sa cyklus líši od jednoduchého opakovaného vykonávania operácií? Ak máte skúsenosti s jazykom Assembly, môžete si pozrieť súbor `.lss` v adresári `simpleserial-base-lab2`. V opačnom prípade tu máte prehľad na úrovni jazyka C o tom, ako (v závislosti od toho, ako to váš kompilátor spracuje) cyklus v skutočnosti vyzerá:

```C
i=0;
start:
    if (i > 20) {
        goto end;
    }    
    A *= 2;
    i += 1;
    goto start;
end:
    trigger_low();
    //...
```

Ako vidíte, mikrokontrolér musí robiť oveľa viac, než len opakovať náš príkaz. Preto kompilátor túto časť kódu často „rozvinie“ (v podstate ju premení späť na pôvodné opakované násobenia). V našom prípade, keďže premenná `i` bola označená ako `volatile`, kompilátor túto optimalizáciu nevykonal. Rozvinutie slučiek tiež zvyšuje veľkosť kódu, takže kompilátor môže rozvinutie vynechať aj v prípade, že optimalizuje s ohľadom na veľkosť kódu.

## Náročné inštrukcie

Doteraz sa všetky inštrukcie, ktoré sme používali (okrem vetvenia v slučke), vykonali za jediný taktový cyklus. Skúsme teraz inštrukciu, ktorej vykonanie trvá viacero taktových cyklov. Bude to závisieť od konkrétnej platformy, ale v prípade zariadenia ARM sa na to dobre hodí inštrukcia delenia. AVR/XMEGA v skutočnosti nemá inštrukciu delenia, takže namiesto nej budete chcieť použiť napríklad inštrukciu násobenia.

In [ ]:
wave4 = capture_trace()
cw.plot(wave4) * cw.plot(wave3)

Pravdepodobne vidíte podobný priebeh ako pri rýchlejšej inštrukcii, ale každé prechádzanie slučkou trvá dlhšie. Dalo by sa tiež očakávať, že dlhšie inštrukcie budú spotrebovávať viac energie. Vidíte to na vašom grafe?

NÁPOVEDA: Na meranie prúdu meria ChipWhisperer pokles napätia na bočnom odpore. To znamená, že priebeh výkonu je v skutočnosti invertovaný (t. j. oblasti s veľkými zápornými výkyvmi predstavujú miesta s vyššou spotrebou energie).

## Závery a ďalšie kroky
Teraz by ste už mali byť celkom presvedčení, že na základe spotreby energie mikrokontroléra môžeme získať určitú predstavu o tom, čo práve robí. V budúcnosti sa budeme zameriavať na objektívnejšie merania, ale to, čo sme urobili v tomto cvičení, má stále veľkú hodnotu. Napríklad implementácie algoritmu AES majú často veľmi charakteristický priebeh. Môžete to využiť na identifikáciu toho, čo neznáme zariadenie robí v rôznych časových okamihoch.

Samozrejme, s týmto laboratórnym cvičením sa dá urobiť oveľa viac. Napríklad, líši sa operácia sčítania od operácie XOR? Nabudúce využijeme to, čo sme sa naučili v tomto laboratórnom cvičení, na prelomenie jednoduchého overovania hesla.

In [18]:
scope.dis()
target.dis()

---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>